# LLM-as-a-Judge: Automated LLM Evaluation

Companion notebook for the [LLM-as-a-Judge wiki page](https://ml-viz-ruby.vercel.app/wiki/llm-as-judge).

We implement G-Eval-style scoring, pairwise comparison, Elo rating aggregation, and positional bias detection — without calling any external API (all simulated).

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)

## 1 — G-Eval: weighted average over score tokens

In [ ]:
def g_eval_score(token_probs):
    """
    G-Eval scoring: weighted average of score tokens {1,2,3,4,5}.
    token_probs: dict mapping score -> probability (from LLM softmax)
    """
    scores = np.array([1, 2, 3, 4, 5])
    probs  = np.array([token_probs.get(s, 0.0) for s in scores])
    probs  = probs / probs.sum()        # normalize
    return float(scores @ probs)

# Simulate LLM output probabilities over score tokens for 3 responses
examples = [
    {"1": 0.02, "2": 0.05, "3": 0.10, "4": 0.43, "5": 0.40},  # high quality
    {"1": 0.30, "2": 0.35, "3": 0.20, "4": 0.10, "5": 0.05},  # low quality
    {"1": 0.05, "2": 0.10, "3": 0.50, "4": 0.25, "5": 0.10},  # medium
]
for i, tp in enumerate(examples):
    score = g_eval_score(tp)
    print(f"Response {i+1}: G-Eval score = {score:.3f}")

## 2 — Elo rating from pairwise comparisons

In [ ]:
def update_elo(Ra, Rb, outcome_a, K=32):
    """
    Update Elo ratings after a matchup.
    outcome_a: 1 if A won, 0 if B won, 0.5 if tie.
    """
    Ea = 1 / (1 + 10**((Rb - Ra)/400))  # expected score for A
    Eb = 1 - Ea
    Ra_new = Ra + K * (outcome_a - Ea)
    Rb_new = Rb + K * ((1 - outcome_a) - Eb)
    return Ra_new, Rb_new

# Simulate 200 pairwise battles between 4 models
n_models = 4
ratings = {f"Model_{i}": 1000.0 for i in range(n_models)}
true_quality = {"Model_0": 0.8, "Model_1": 0.6, "Model_2": 0.5, "Model_3": 0.3}

models = list(ratings.keys())
for _ in range(200):
    a, b = rng.choice(models, 2, replace=False)
    # Simulate battle: better model wins with prob proportional to quality difference
    p_a = true_quality[a] / (true_quality[a] + true_quality[b])
    outcome = 1.0 if rng.random() < p_a else 0.0
    ratings[a], ratings[b] = update_elo(ratings[a], ratings[b], outcome)

print("Final Elo ratings (sorted):")
for m, r in sorted(ratings.items(), key=lambda x: -x[1]):
    print(f"  {m}: Elo={r:.1f}  (true quality={true_quality[m]})")

## 3 — Positional bias detection

In [ ]:
def detect_positional_bias(n_trials=200):
    """
    Simulate pairwise comparison where the judge has positional bias:
    it prefers the FIRST response with probability p_bias.
    Measure how often Position A wins vs Position B.
    """
    p_bias = 0.65  # judge prefers first response 65% of the time regardless of quality
    wins_a, wins_b = 0, 0
    for _ in range(n_trials):
        # Equally good responses (no true quality difference)
        judge_picks_first = rng.random() < p_bias
        # We randomize which model goes first
        a_is_first = rng.random() < 0.5
        if judge_picks_first:
            if a_is_first: wins_a += 1
            else: wins_b += 1
        else:
            if a_is_first: wins_b += 1
            else: wins_a += 1
    return wins_a / n_trials, wins_b / n_trials

rate_a, rate_b = detect_positional_bias(500)
print(f"Win rate A: {rate_a:.3f}  Win rate B: {rate_b:.3f}")
print(f"Expected without bias: 0.500  Difference: {abs(rate_a - 0.5):.3f}")
print("\nFix: always evaluate BOTH orderings (A vs B and B vs A) and average.")

## ✏️ Your turn

In [ ]:
def pairwise_win_rate_matrix(models, quality_scores, n_battles=100):
    """
    Simulate all pairwise battles and return a win-rate matrix.
    models: list of model names
    quality_scores: dict of model -> true quality (0-1)
    n_battles: battles per pair
    
    Returns: (n, n) matrix where matrix[i,j] = win rate of model i vs model j
    """
    n = len(models)
    # TODO(you): for each (i, j) pair, simulate n_battles and record win rate
    return ...

models = ["GPT4o", "Claude3.5", "Llama70B", "Mistral7B"]
quality = {"GPT4o": 0.85, "Claude3.5": 0.82, "Llama70B": 0.65, "Mistral7B": 0.50}
W = pairwise_win_rate_matrix(models, quality, n_battles=50)
if W is not None:
    print("Win rate matrix (row beats column):")
    print(f"{'':12s}", " ".join(f"{m[:7]:>8s}" for m in models))
    for i, m in enumerate(models):
        print(f"{m[:12]:12s}", " ".join(f"{W[i,j]:>8.3f}" for j in range(len(models))))

<details><summary>Solution</summary>

```python
def pairwise_win_rate_matrix(models, quality_scores, n_battles=100):
    n = len(models)
    W = np.zeros((n, n))
    for i, a in enumerate(models):
        for j, b in enumerate(models):
            if i == j:
                W[i,j] = 0.5
                continue
            p_a = quality_scores[a] / (quality_scores[a] + quality_scores[b])
            wins = rng.binomial(n_battles, p_a)
            W[i,j] = wins / n_battles
    return W
```
</details>